# BrewBeats — Mini Project 2: Section A (Spark RDD & DataFrames)
**Folder:** `Section A/`  
**Stack:** Apache Spark (PySpark RDDs & DataFrames)

---
## Section A - Q1: Load `Spotify_data.csv` into an RDD and count the number of songs per artist.

In [1]:
# === Section A - Q1: Load Spotify_data.csv into an RDD and count songs per artist ===
from pyspark.sql import SparkSession
import csv

# Start a local Spark session using all CPU cores ("local[*]")
# To work with RDDs, we need the SparkContext, which we get from spark.sparkContext (sc)
spark = (
    SparkSession.builder
    .appName("BrewBeats_MiniProject2_Spark")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")  # Hide Spark's long warning logs

# Load Spotify_data.csv into an RDD as raw text lines (each item in the RDD is one row string)
spotify_rdd = sc.textFile("Spotify_data.csv")

# Grab the first row (the header with column names) and filter it out so we only process actual songs.
# Then use csv.reader inside .map() to split each row by commas (this prevents errors when a song title
# has a comma inside quotes, like "Me, Myself & I").
header = spotify_rdd.first()
parsed_spotify_rdd = (
    spotify_rdd
    .filter(lambda line: line != header)
    .map(lambda line: next(csv.reader([line])))
)

# Count songs per artist:
# 1. .map() picks column 16 (the artist name) and pairs it with the number 1 -> (artist, 1)
# 2. .reduceByKey() groups by artist name and adds all the 1s together (a + b) to get their total songs
# 3. .sortBy() sorts by the song count (pair[1]) from highest to lowest (ascending=False)
songs_per_artist_rdd = (
    parsed_spotify_rdd
    .map(lambda cols: (cols[16].strip(), 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda pair: pair[1], ascending=False)
)

print("=== Section A - Q1: Number of Songs per Artist (RDD) ===")
print(f"Total Distinct Artists: {songs_per_artist_rdd.count()}\n")
print("Top 15 Artists by Song Count:")
print("-" * 35)
for artist, count in songs_per_artist_rdd.take(15):
    print(f"{artist:<25} | {count} songs")

=== Section A - Q1: Number of Songs per Artist (RDD) ===
Total Distinct Artists: 1343

Top 15 Artists by Song Count:
-----------------------------------
Drake                     | 16 songs
Rick Ross                 | 13 songs
Disclosure                | 12 songs
Backstreet Boys           | 10 songs
WALK THE MOON             | 10 songs
Crystal Castles           | 9 songs
FIDLAR                    | 9 songs
Future                    | 8 songs
Kanye West                | 8 songs
Fall Out Boy              | 8 songs
Demi Lovato               | 8 songs
Big Time Rush             | 8 songs
*NSYNC                    | 8 songs
Skrillex                  | 8 songs
Kina Grannis              | 8 songs


---
## Section A - Q2: Using RDD transformations, find the average danceability across all songs.

In [2]:
# === Section A - Q2: Average danceability across all songs using RDD transformations ===

# Column index 2 in Spotify_data.csv is 'danceability'.
# To calculate the average with RDD transformations:
# 1. .map() converts each row into a pair: (danceability_score_as_float, 1_song)
# 2. .reduce() adds up all the danceability scores (a[0] + b[0]) AND counts all the songs (a[1] + b[1])
total_danceability, total_songs = (
    parsed_spotify_rdd
    .map(lambda cols: (float(cols[2]), 1))
    .reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))
)

# Divide the total danceability sum by the total number of songs to get the average
avg_danceability = total_danceability / total_songs

print("=== Section A - Q2: Average Danceability Across All Songs (RDD) ===")
print(f"Total Songs Processed : {total_songs}")
print(f"Sum of Danceability   : {total_danceability:.4f}")
print(f"Average Danceability  : {avg_danceability:.4f} ({avg_danceability * 100:.2f}%)")

=== Section A - Q2: Average Danceability Across All Songs (RDD) ===
Total Songs Processed : 2017
Sum of Danceability   : 1247.3570
Average Danceability  : 0.6184 (61.84%)
